In [4]:
import pandas as pd

# Replace 'yourfile.csv' with your actual file name
df = pd.read_csv('Students Social Media Addiction.csv')

# Display the first few rows
df.head()


,Student_ID,Age,Gender,Academic_Level,Country,Avg_Daily_Usage_Hours,Most_Used_Platform,Affects_Academic_Performance,Sleep_Hours_Per_Night,Mental_Health_Score,Relationship_Status,Conflicts_Over_Social_Media,Addicted_Score
0,1,19,Female,Undergraduate,Bangladesh,5.2,Instagram,Yes,6.5,6,In Relationship,3,8
1,2,22,Male,Graduate,India,2.1,Twitter,No,7.5,8,Single,0,3
2,3,20,Female,Undergraduate,USA,6.0,TikTok,Yes,5.0,5,Complicated,4,9
3,4,18,Male,High School,UK,3.0,YouTube,No,7.0,7,Single,1,4
4,5,21,Male,Graduate,Canada,4.5,Facebook,Yes,6.0,6,In Relationship,2,7


In [5]:
df.drop(columns=["Student_ID"], inplace=True)


In [6]:
from sklearn.preprocessing import LabelEncoder

label_cols = ['Gender', 'Academic_Level', 'Country', 'Most_Used_Platform',
              'Affects_Academic_Performance', 'Relationship_Status']

label_encoders = {}
for col in label_cols:
    le = LabelEncoder()
    df[col] = le.fit_transform(df[col].astype(str))
    label_encoders[col] = le


In [7]:
df.head()

,Age,Gender,Academic_Level,Country,Avg_Daily_Usage_Hours,Most_Used_Platform,Affects_Academic_Performance,Sleep_Hours_Per_Night,Mental_Health_Score,Relationship_Status,Conflicts_Over_Social_Media,Addicted_Score
0,19,0,2,10,5.2,1,1,6.5,6,1,3,8
1,22,1,0,39,2.1,7,0,7.5,8,2,0,3
2,20,0,2,102,6.0,6,1,5.0,5,0,4,9
3,18,1,1,101,3.0,11,0,7.0,7,2,1,4
4,21,1,0,18,4.5,0,1,6.0,6,1,2,7


In [8]:
# Binary classification target
df['Mental_Health_Binary'] = df['Mental_Health_Score'].apply(lambda x: 1 if x <= 5 else 0)

# Multi-class target: 0 = Low, 1 = Medium, 2 = High
def multi_class_label(score):
    if score <= 4:
        return 0
    elif score <= 7:
        return 1
    else:
        return 2

df['Mental_Health_Multiclass'] = df['Mental_Health_Score'].apply(multi_class_label)


In [9]:
from sklearn.model_selection import train_test_split

# Features (X) and Targets (y)
X = df.drop(columns=['Mental_Health_Score', 'Mental_Health_Binary', 'Mental_Health_Multiclass'])
y_binary = df['Mental_Health_Binary']
y_multi = df['Mental_Health_Multiclass']

# Split for binary classification
X_train_bin, X_test_bin, y_train_bin, y_test_bin = train_test_split(X, y_binary, test_size=0.2, random_state=42)

# Split for multi-class classification
X_train_multi, X_test_multi, y_train_multi, y_test_multi = train_test_split(X, y_multi, test_size=0.2, random_state=42)


In [11]:
import pandas as pd
import numpy as np
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.naive_bayes import GaussianNB
from sklearn.svm import SVC
from sklearn.neural_network import MLPClassifier
from sklearn.metrics import f1_score, accuracy_score, precision_score, recall_score
from sklearn.preprocessing import StandardScaler
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import LSTM, Dense, Dropout

# Evaluation function with zero_division handling
def get_metrics(y_true, y_pred):
    return {
        'F1-Score': f1_score(y_true, y_pred, zero_division=0),
        'Accuracy': accuracy_score(y_true, y_pred),
        'Recall': recall_score(y_true, y_pred, zero_division=0),
        'Precision': precision_score(y_true, y_pred, zero_division=0)
    }


In [12]:
rf = RandomForestClassifier()
rf.fit(X_train_bin, y_train_bin)
y_pred_rf = rf.predict(X_test_bin)


In [13]:
lr = LogisticRegression(max_iter=1000)
lr.fit(X_train_bin, y_train_bin)
y_pred_lr = lr.predict(X_test_bin)

nb = GaussianNB()
nb.fit(X_train_bin, y_train_bin)
y_pred_nb = nb.predict(X_test_bin)


scaler = StandardScaler()
X_train_svm = scaler.fit_transform(X_train_bin)
X_test_svm = scaler.transform(X_test_bin)

svm = SVC(class_weight='balanced')  # Optional: class_weight handles imbalance
svm.fit(X_train_svm, y_train_bin)
y_pred_svm = svm.predict(X_test_svm)

X_train_mlp = scaler.fit_transform(X_train_bin)
X_test_mlp = scaler.transform(X_test_bin)

mlp = MLPClassifier(hidden_layer_sizes=(64, 32), max_iter=300)
mlp.fit(X_train_mlp, y_train_bin)
y_pred_mlp = mlp.predict(X_test_mlp)



In [14]:
# Reshape for LSTM input: (samples, time steps, features)
X_train_lstm = np.reshape(X_train_mlp, (X_train_mlp.shape[0], 1, X_train_mlp.shape[1]))
X_test_lstm = np.reshape(X_test_mlp, (X_test_mlp.shape[0], 1, X_test_mlp.shape[1]))

# Build model
model = Sequential()
model.add(LSTM(64, input_shape=(1, X_train_mlp.shape[1])))
model.add(Dropout(0.5))
model.add(Dense(1, activation='sigmoid'))
model.compile(loss='binary_crossentropy', optimizer='adam', metrics=['accuracy'])

# Train
model.fit(X_train_lstm, y_train_bin, epochs=10, batch_size=32, verbose=0)

# Predict
y_pred_lstm = (model.predict(X_test_lstm) > 0.5).astype("int32").flatten()


C:\Users\dipra\AppData\Roaming\Python\Python312\site-packages\keras\src\layers\rnn\rnn.py:199: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(**kwargs)


5/5 ━━━━━━━━━━━━━━━━━━━━ 0s 38ms/step


In [15]:
results = {}
results['Random Forest'] = get_metrics(y_test_bin, y_pred_rf)
results['Logistic Regression'] = get_metrics(y_test_bin, y_pred_lr)
results['Naive Bayes'] = get_metrics(y_test_bin, y_pred_nb)
results['SVM'] = get_metrics(y_test_bin, y_pred_svm)
results['MLP'] = get_metrics(y_test_bin, y_pred_mlp)
results['LSTM'] = get_metrics(y_test_bin, y_pred_lstm)

# Display as formatted table
results_df = pd.DataFrame(results).T
print(results_df)


                     F1-Score  Accuracy    Recall  Precision
Random Forest        0.962963  0.978723  0.928571   1.000000
Logistic Regression  0.857143  0.921986  0.785714   0.942857
Naive Bayes          0.678261  0.737589  0.928571   0.534247
SVM                  0.906977  0.943262  0.928571   0.886364
MLP                  0.952381  0.971631  0.952381   0.952381
LSTM                 0.761905  0.858156  0.761905   0.761905


In [16]:
pd.DataFrame(results).T

,F1-Score,Accuracy,Recall,Precision
Random Forest,0.962963,0.978723,0.928571,1.000000
Logistic Regression,0.857143,0.921986,0.785714,0.942857
Naive Bayes,0.678261,0.737589,0.928571,0.534247
SVM,0.906977,0.943262,0.928571,0.886364
MLP,0.952381,0.971631,0.952381,0.952381
LSTM,0.761905,0.858156,0.761905,0.761905


For Multiclass

In [18]:
from sklearn.tree import DecisionTreeClassifier
from sklearn.neighbors import KNeighborsClassifier
import lightgbm as lgb
from xgboost import XGBClassifier
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, Conv1D, MaxPooling1D, Flatten, Dropout
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score
from sklearn.preprocessing import StandardScaler
import numpy as np
import pandas as pd


In [19]:
def get_metrics_multi(y_true, y_pred):
    return {
        'F1-Score': f1_score(y_true, y_pred, average='macro', zero_division=0),
        'Accuracy': accuracy_score(y_true, y_pred),
        'Recall': recall_score(y_true, y_pred, average='macro', zero_division=0),
        'Precision': precision_score(y_true, y_pred, average='macro', zero_division=0)
    }


In [20]:
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train_multi)
X_test_scaled = scaler.transform(X_test_multi)


In [21]:
dt = DecisionTreeClassifier()
dt.fit(X_train_multi, y_train_multi)
y_pred_dt = dt.predict(X_test_multi)

knn = KNeighborsClassifier(n_neighbors=3)
knn.fit(X_train_scaled, y_train_multi)
y_pred_knn = knn.predict(X_test_scaled)

lgbm = lgb.LGBMClassifier(
    n_estimators=100,
    learning_rate=0.1,
    num_leaves=20,
    min_data_in_leaf=3,
    min_gain_to_split=0,
    random_state=42
)
lgbm.fit(X_train_multi, y_train_multi)
y_pred_lgb = lgbm.predict(X_test_multi)

xgb = XGBClassifier(use_label_encoder=False, eval_metric='mlogloss')
xgb.fit(X_train_multi, y_train_multi)
y_pred_xgb = xgb.predict(X_test_multi)

xgb = XGBClassifier(use_label_encoder=False, eval_metric='mlogloss')
xgb.fit(X_train_multi, y_train_multi)
y_pred_xgb = xgb.predict(X_test_multi)







[LightGBM] [Warning] min_data_in_leaf is set=3, min_child_samples=20 will be ignored. Current value: min_data_in_leaf=3
[LightGBM] [Warning] min_gain_to_split is set=0, min_split_gain=0.0 will be ignored. Current value: min_gain_to_split=0
[LightGBM] [Warning] min_data_in_leaf is set=3, min_child_samples=20 will be ignored. Current value: min_data_in_leaf=3
[LightGBM] [Warning] min_gain_to_split is set=0, min_split_gain=0.0 will be ignored. Current value: min_gain_to_split=0
[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.000145 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 197
[LightGBM] [Info] Number of data points in the train set: 564, number of used features: 11
[LightGBM] [Info] Start training from score -3.116178
[LightGBM] [Info] Start training from score -0.219162
[LightGBM] [Info] Start training from score -1.880707
[LightGBM] [Warning] No further splits with positive gain, best gain: -i

C:\Users\dipra\AppData\Roaming\Python\Python312\site-packages\xgboost\training.py:183: UserWarning: [12:22:23] WARNING: C:\actions-runner\_work\xgboost\xgboost\src\learner.cc:738: 
Parameters: { "use_label_encoder" } are not used.

  bst.update(dtrain, iteration=i, fobj=obj)
C:\Users\dipra\AppData\Roaming\Python\Python312\site-packages\xgboost\training.py:183: UserWarning: [12:22:23] WARNING: C:\actions-runner\_work\xgboost\xgboost\src\learner.cc:738: 
Parameters: { "use_label_encoder" } are not used.

  bst.update(dtrain, iteration=i, fobj=obj)


In [23]:
# Reshape to 3D for CNN: (samples, timesteps, features)
X_train_cnn = X_train_scaled.reshape(X_train_scaled.shape[0], X_train_scaled.shape[1], 1)
X_test_cnn = X_test_scaled.reshape(X_test_scaled.shape[0], X_test_scaled.shape[1], 1)

cnn = Sequential()
cnn.add(Conv1D(64, kernel_size=3, activation='relu', input_shape=(X_train_cnn.shape[1], 1)))
cnn.add(MaxPooling1D(pool_size=2))
cnn.add(Flatten())
cnn.add(Dense(128, activation='relu'))
cnn.add(Dropout(0.5))
cnn.add(Dense(3, activation='softmax'))  # 3 classes

cnn.compile(loss='sparse_categorical_crossentropy', optimizer='adam', metrics=['accuracy'])
cnn.fit(X_train_cnn, y_train_multi, epochs=10, batch_size=32, verbose=0)

y_pred_cnn = np.argmax(cnn.predict(X_test_cnn), axis=1)


C:\Users\dipra\AppData\Roaming\Python\Python312\site-packages\keras\src\layers\convolutional\base_conv.py:113: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


5/5 ━━━━━━━━━━━━━━━━━━━━ 0s 21ms/step


In [24]:
results_multi = {}
results_multi['Decision Tree'] = get_metrics_multi(y_test_multi, y_pred_dt)
results_multi['KNN'] = get_metrics_multi(y_test_multi, y_pred_knn)
results_multi['LightGBM'] = get_metrics_multi(y_test_multi, y_pred_lgb)
results_multi['XGBoost'] = get_metrics_multi(y_test_multi, y_pred_xgb)
results_multi['CNN'] = get_metrics_multi(y_test_multi, y_pred_cnn)

# Final Table
results_multi_df = pd.DataFrame(results_multi).T
print(results_multi_df)

               F1-Score  Accuracy    Recall  Precision
Decision Tree  0.856937  0.957447  0.902422   0.820239
KNN            0.833384  0.964539  0.810969   0.863748
LightGBM       0.886196  0.964539  0.905271   0.870264
XGBoost        0.877668  0.957447  0.902422   0.858161
CNN            0.554790  0.900709  0.541453   0.574833


In [25]:
pd.DataFrame(results_multi).T

,F1-Score,Accuracy,Recall,Precision
Decision Tree,0.856937,0.957447,0.902422,0.820239
KNN,0.833384,0.964539,0.810969,0.863748
LightGBM,0.886196,0.964539,0.905271,0.870264
XGBoost,0.877668,0.957447,0.902422,0.858161
CNN,0.554790,0.900709,0.541453,0.574833
